# Task 2 — ID-Block Model (uses the `player_id` leak)

**This notebook contains only the ID method.** The features-only models are in `04_model_no_leak.ipynb`.

`player_id` numbers were issued in blocks by segment, so the ID alone predicts the label. This is **target leakage**: it comes from how the dataset was built and would not work for real new players. It is kept separate so that the two approaches can be reported independently.

1. Audit: is the label related to the ID?
2. ID-block classifier: 5-fold CV
3. Why the score stops near 0.97: label noise
4. Fit on all training data → write the submission

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
from sklearn.metrics import classification_report

from data import load_train, load_test, TARGET, CLASS_NAMES
from evaluate import cv_fit_in_fold, cv_model, get_folds
from id_model import IDBlockClassifier, id_number
from model_zoo import VIEWS
from models import AvgHistNB
from submission import save_submission

ORDER = list(CLASS_NAMES.values())
train, test = load_train(), load_test()
y = train[TARGET].to_numpy()
folds = get_folds(y)

## 1. Audit: is the label related to the ID?

In [2]:
pid = id_number(train["player_id"])
display(pd.crosstab(pd.qcut(pid, 10), train[TARGET].map(CLASS_NAMES).values, normalize="index")[ORDER].round(3))

col_0,Casual,Grinder,Social,Whale
row_0,,,,
"(-0.001, 13014.8]",0.967,0.012,0.011,0.010
"(13014.8, 26011.8]",0.970,0.011,0.011,0.008
"(26011.8, 38984.7]",0.967,0.011,0.010,0.012
"(38984.7, 51991.6]",0.971,0.010,0.009,0.010
"(51991.6, 65028.5]",0.010,0.970,0.010,0.010
"(65028.5, 78055.4]",0.012,0.970,0.009,0.009
"(78055.4, 90986.3]",0.010,0.491,0.489,0.010
"(90986.3, 104041.6]",0.010,0.009,0.974,0.007
"(104041.6, 117067.1]",0.010,0.009,0.488,0.493


Sorted by ID, the segments appear in contiguous blocks. The file looks random only because its rows are shuffled.

## 2. ID-block classifier

In [3]:
id_cols = train[["player_id"]]
print("ID-block model ", end="")
oof, score, _ = cv_model(IDBlockClassifier(), id_cols, y, folds)
pred = oof.argmax(1)
print(classification_report(y, pred, target_names=ORDER, digits=4))

ID-block model 

macro F1 0.9677  (folds 0.9677 ± 0.0009)  [1 feats, 0s]
              precision    recall  f1-score   support

      Casual     0.9686    0.9845    0.9765     40934
     Grinder     0.9702    0.9692    0.9697     26034
      Social     0.9719    0.9605    0.9662     21014
       Whale     0.9706    0.9466    0.9585     16018

    accuracy                         0.9700    104000
   macro avg     0.9703    0.9652    0.9677    104000
weighted avg     0.9700    0.9700    0.9699    104000



The boundaries are learned from the training labels by choosing the cut points that maximise accuracy. They are not hard-coded.

## 3. Why the score stops near 0.97
About 3% of labels disagree with the ID block. Two checks show what those rows are.

In [4]:
off = pred != y
print(f"label differs from ID block: {off.sum():,} rows ({off.mean():.1%})")
display(pd.crosstab(pd.Series(pred[off], name="ID block").map(CLASS_NAMES),
                    pd.Series(y[off], name="label").map(CLASS_NAMES))[ORDER])

nb_oof, _, _ = cv_fit_in_fold(AvgHistNB(), train, y, VIEWS["nb"], folds, verbose=False)
nb = nb_oof.argmax(1)
print(f"on those rows, a features-only model agrees with the LABEL {np.mean(nb[off] == y[off]):.1%} "
      f"and with the ID BLOCK {np.mean(nb[off] == pred[off]):.1%}")

label differs from ID block: 3,122 rows (3.0%)


label,Casual,Grinder,Social,Whale
ID block,,,,
Casual,0,453,432,420
Grinder,283,0,246,246
Social,206,188,0,189
Whale,146,160,153,0


on those rows, a features-only model agrees with the LABEL 6.9% and with the ID BLOCK 78.3%


- The wrong labels are spread **evenly** over the other three classes.
- On those rows the player's features match the **ID block**, not the label.

So the labels were most likely flipped at random after the features were generated. Nothing can predict them, which sets the ceiling at about 97% accuracy.

## 4. Fit on all training data → submission

In [5]:
model = IDBlockClassifier().fit(train, y)
print("learned boundaries:", model.bounds_)
test_pred = model.predict(test)
display(pd.Series(test_pred).map(CLASS_NAMES).value_counts(normalize=True).loc[ORDER].round(3).to_frame("test share"))
full = save_submission(test_pred, "v2_idblock")
print("saved submissions/task2_v2_idblock.csv and submissions/full_v2_idblock.csv")
full.head()

learned boundaries: [ 51999.5  84499.  110499.5]


,test share
Casual,0.400
Grinder,0.250
Social,0.201
Whale,0.149


saved submissions/task2_v2_idblock.csv and submissions/full_v2_idblock.csv


,id,task1,task2,task3,task4,task5
0,ANS00001,1,2,0.0,0,1
1,ANS00002,1,0,0.0,0,1
2,ANS00003,1,3,0.0,0,1
3,ANS00004,1,2,0.0,0,1
4,ANS00005,1,1,0.0,0,1
